# 🔬 离线调试：`agent/run` — 走完整生产路径，无需任何 Server

**生产路径完整复现**：

```
RunAgentInput (原始 payload)
      ↓
_inject_state()          ← 解析 headers / context → 注入 org_name, layer, project_slug
      ↓
LangGraphAGUIAgent.run() ← ag-ui message 转换 + MemorySaver + event stream
      ↓
admin_graph.ainvoke      ← LangGraph
```

只 mock 一层：`agents.shared.make_client`（GraphQL → fake data）  
HTTP headers 通过 `MagicMock` 注入，完全无需启动 FastAPI server。

In [10]:
import sys, os, json, uuid

AGENT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if AGENT_ROOT not in sys.path:
    sys.path.insert(0, AGENT_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(AGENT_ROOT, '.env'))

from logging_setup import setup_logging
setup_logging()

import config
print('AGENT_ROOT:', AGENT_ROOT)
print('LLM_MODEL :', config.LLM_MODEL)

AGENT_ROOT: /data/home/lukemxjia/modelcraft/modelcraft-agent
LLM_MODEL : deepseek-chat


## 1. Mock GraphQL Client

In [11]:
from unittest.mock import AsyncMock, MagicMock, patch

FAKE_PROJECTS = [
    {"id": "p1", "slug": "onboardceshi", "title": "onboard测试", "description": "", "status": "ACTIVE"},
    {"id": "p2", "slug": "abcde",        "title": "abcde",       "description": "", "status": "ACTIVE"},
    {"id": "p3", "slug": "pro1",         "title": "pro1",        "description": "", "status": "ACTIVE"},
]

def make_fake_client(state=None):
    c = MagicMock()
    c.list_projects    = AsyncMock(return_value={"data": {"projects": FAKE_PROJECTS}})
    c.list_databases   = AsyncMock(return_value={"data": {"listDatabases": {"edges": [
        {"node": {"name": "maindb"}}, {"node": {"name": "analyticsdb"}}
    ]}}})
    c.list_models      = AsyncMock(return_value={"data": {"models": {"items": [
        {"id": "m1", "name": "user",  "title": "用户"},
        {"id": "m2", "name": "order", "title": "订单"},
    ], "hasNextPage": False}}})
    c.get_model_fields = AsyncMock(return_value={"data": {"model": {"fields": []}}})
    c.query_model      = AsyncMock(return_value={"data": {"queryModel": {"items": [], "hasNextPage": False}}})
    c.nl2filter        = AsyncMock(return_value={"data": {"nl2filter": {"filter": "{}", "explanation": "mock"}}})
    return c

_patcher = patch('agents.shared.make_client', side_effect=make_fake_client)
_patcher.start()
print('✅ make_client 已 mock')

✅ make_client 已 mock


## 2. 构造 RunAgentInput + 伪造 Request headers

原始 `agent/run` payload 直接映射到 `RunAgentInput`。  
`_inject_state` 从 HTTP headers 读 `X-Org-Name` / `Authorization`，用 `MagicMock` 伪造即可。

In [12]:
# ── 取消注释以完全离线（mock LLM 直接返回一次 tool_call + 一次文本回复）──

# import json
# from unittest.mock import patch, AsyncMock
# from langchain_core.messages import AIMessage

# _TOOL_CALL_RESPONSE = AIMessage(
#     content="",
#     tool_calls=[{"name": "list_projects", "args": {}, "id": "mock-call-1", "type": "tool_call"}],
# )
# _TEXT_RESPONSE = AIMessage(
#     content="你有以下项目：\n- onboard测试 (onboardceshi)\n- abcde\n- pro1"
# )

# _call_count = {"n": 0}
# async def _mock_llm_invoke(messages, **kwargs):
#     _call_count["n"] += 1
#     return _TOOL_CALL_RESPONSE if _call_count["n"] == 1 else _TEXT_RESPONSE

# _llm_patcher = patch('agents.admin_agent.ChatOpenAI')
# mock_llm_cls = _llm_patcher.start()
# mock_llm_cls.return_value.bind_tools.return_value.ainvoke = _mock_llm_invoke
# print('✅ LLM 已 mock，完全离线')

print('ℹ️  LLM mock 未启用，使用 .env 配置的真实 LLM')

ℹ️  LLM mock 未启用，使用 .env 配置的真实 LLM


## 3. 构造 State（还原原始 `agent/run` 请求）

In [13]:
import uuid
from langchain_core.messages import HumanMessage

# ── 从原始 agent/run 请求的 forwardedProps + context 中提取 ──────────
# forwardedProps: { projectId: 'default', projectSlug: 'Default Project', orgName: 'lukeco' }
# context[2].value: { layer: 'org', orgName: 'lukeco', ... }

state = {
    "messages"     : [HumanMessage(content="列出我所有的项目")],
    "authorization": "Bearer offline-debug-token",   # 不走 server，随便填
    "org_name"     : "lukeco",
    "layer"        : "org",              # 来自 context[2].value.layer
    "project_slug" : "Default Project", # 来自 forwardedProps.projectSlug
    "current_model": "",
    "current_db"   : "",
}

cfg = {"configurable": {"thread_id": str(uuid.uuid4())}}

print('State ready:')
for k, v in state.items():
    if k != 'messages':
        print(f'  {k:15s}: {v}')
print(f'  {"messages":15s}: [{state["messages"][0].content!r}]')
print('thread_id:', cfg['configurable']['thread_id'])

State ready:
  authorization  : Bearer offline-debug-token
  org_name       : lukeco
  layer          : org
  project_slug   : Default Project
  current_model  : 
  current_db     : 
  messages       : ['列出我所有的项目']
thread_id: 899fe77b-338a-4b44-a57c-0f584689e63e


## 4. 运行 Agent（ainvoke）

In [14]:
from agents.admin_agent import admin_graph

result = await admin_graph.ainvoke(state, config=cfg)

reply = result['messages'][-1].content
print('─' * 60)
print('🤖 Agent 回复：')
print(reply)
print('─' * 60)

{"service": "modelcraft-agent", "tool_name": "list_projects", "args_summary": "org=lukeco", "event": "tool.call.start", "level": "info", "timestamp": "2026-05-20T17:24:59.855333Z"}
{"service": "modelcraft-agent", "tool_name": "list_projects", "duration_ms": 0.08, "success": true, "event": "tool.call.end", "level": "info", "timestamp": "2026-05-20T17:24:59.856030Z"}
────────────────────────────────────────────────────────────
🤖 Agent 回复：
您当前组织（lukeco）下共有 **3 个项目**：

| 项目名称 | 标识（slug） | 状态 |
|---------|------------|------|
| 📁 **onboard测试** | `onboardceshi` | ✅ 活跃 |
| 📁 **abcde** | `abcde` | ✅ 活跃 |
| 📁 **pro1** | `pro1` | ✅ 活跃 |

请问您想进入哪个项目进行操作？
────────────────────────────────────────────────────────────


## 5. 检查完整消息轨迹（tool_call → tool_result → 回复）

In [15]:
from langchain_core.messages import AIMessage, ToolMessage

print(f'共 {len(result["messages"])} 条消息：\n')
for i, msg in enumerate(result['messages']):
    role = type(msg).__name__
    if isinstance(msg, AIMessage) and getattr(msg, 'tool_calls', None):
        calls = [f"{tc['name']}({tc['args']})" for tc in msg.tool_calls]
        print(f'[{i}] {role:15s} → tool_calls: {", ".join(calls)}')
    elif isinstance(msg, ToolMessage):
        preview = str(msg.content)[:120].replace('\n', ' ')
        print(f'[{i}] {role:15s} [{msg.name}] → {preview}...')
    else:
        preview = str(msg.content)[:80].replace('\n', ' ')
        print(f'[{i}] {role:15s} → {preview}')

共 4 条消息：

[0] HumanMessage    → 列出我所有的项目
[1] AIMessage       → tool_calls: list_projects({})
[2] ToolMessage     [list_projects] → [{"id": "p1", "slug": "onboardceshi", "title": "onboard测试", "description": "", "status": "ACTIVE"}, {"id": "p2", "slug":...
[3] AIMessage       → 您当前组织（lukeco）下共有 **3 个项目**：  | 项目名称 | 标识（slug） | 状态 | |---------|------------|--


## 6. 流式模式（实时看每步）

In [16]:
import uuid

stream_state = dict(state)
stream_state['messages'] = [HumanMessage(content="列出我所有的项目")]
stream_cfg = {"configurable": {"thread_id": str(uuid.uuid4())}}

print('── Streaming ──\n')
async for event in admin_graph.astream_events(stream_state, config=stream_cfg, version='v2'):
    kind = event['event']
    name = event.get('name', '')

    if kind == 'on_chat_model_stream':
        chunk = event['data']['chunk']
        if hasattr(chunk, 'content') and chunk.content:
            print(chunk.content, end='', flush=True)

    elif kind == 'on_tool_start':
        inp = event['data'].get('input', {})
        print(f'\n\n▶ tool [{name}]  input={inp}')

    elif kind == 'on_tool_end':
        out = str(event['data'].get('output', ''))[:200]
        print(f'◀ tool [{name}]  output={out}')

print('\n\n── 完成 ──')

── Streaming ──

好的，我来查看您组织下的所有项目。

▶ tool [list_projects]  input={}
{"service": "modelcraft-agent", "tool_name": "list_projects", "args_summary": "org=lukeco", "event": "tool.call.start", "level": "info", "timestamp": "2026-05-20T17:25:02.145932Z"}
{"service": "modelcraft-agent", "tool_name": "list_projects", "duration_ms": 0.07, "success": true, "event": "tool.call.end", "level": "info", "timestamp": "2026-05-20T17:25:02.146361Z"}
◀ tool [list_projects]  output=content='[{"id": "p1", "slug": "onboardceshi", "title": "onboard测试", "description": "", "status": "ACTIVE"}, {"id": "p2", "slug": "abcde", "title": "abcde", "description": "", "status": "ACTIVE"}, {"i
您当前组织（lukeco）下共有 **3 个项目**：

| 项目名称 | 标识（slug） | 状态 |
|---------|------------|------|
| 📁 **onboard测试** | `onboardceshi` | ✅ 活跃 |
| 📁 **abcde** | `abcde` | ✅ 活跃 |
| 📁 **pro1** | `pro1` | ✅ 活跃 |

请问您想进入哪个项目进行操作？

── 完成 ──


## 7. 换用不同用户消息快速测试

In [17]:
import uuid

TEST_MESSAGES = [
    "列出我所有的项目",
    "帮我进入 abcde 项目",
    "pro1 项目里有哪些模型",
    "新建一个项目，名称叫 test-offline",
]

async def quick_test(msg: str, layer="org", project_slug="") -> str:
    s = dict(state)
    s['messages']      = [HumanMessage(content=msg)]
    s['layer']         = layer
    s['project_slug']  = project_slug
    cfg2 = {"configurable": {"thread_id": str(uuid.uuid4())}}
    r = await admin_graph.ainvoke(s, config=cfg2)
    reply = r['messages'][-1].content

    # 统计 tool calls
    tools_used = [
        tc['name']
        for m in r['messages'] if isinstance(m, AIMessage)
        for tc in getattr(m, 'tool_calls', [])
    ]
    print(f"Q: {msg!r}")
    print(f"   tools: {tools_used or ['(none)']}") 
    print(f"   A: {reply[:100]}\n")
    return reply

for msg in TEST_MESSAGES:
    await quick_test(msg)

{"service": "modelcraft-agent", "tool_name": "list_projects", "args_summary": "org=lukeco", "event": "tool.call.start", "level": "info", "timestamp": "2026-05-20T17:25:04.504321Z"}
{"service": "modelcraft-agent", "tool_name": "list_projects", "duration_ms": 0.07, "success": true, "event": "tool.call.end", "level": "info", "timestamp": "2026-05-20T17:25:04.505018Z"}
Q: '列出我所有的项目'
   tools: ['list_projects']
   A: 您当前组织 **lukeco** 下共有 **3 个项目**：

| 项目名称 | 标识 (slug) | 状态 |
|---------|------------|------|
| 📁 **onb

{"service": "modelcraft-agent", "tool_name": "list_projects", "args_summary": "org=lukeco", "event": "tool.call.start", "level": "info", "timestamp": "2026-05-20T17:25:06.971893Z"}
{"service": "modelcraft-agent", "tool_name": "list_projects", "duration_ms": 0.07, "success": true, "event": "tool.call.end", "level": "info", "timestamp": "2026-05-20T17:25:06.972572Z"}
Q: '帮我进入 abcde 项目'
   tools: ['list_projects', 'navigate_to_project']
   A: 好的，已确认 **abcde** 项目存在（slug: `abcde`）。我

## 8. 清理 Mock（结束调试后运行）

In [18]:
_patcher.stop()
# _llm_patcher.stop()  # 如果启用了 LLM mock
print('✅ 所有 mock 已恢复')

✅ 所有 mock 已恢复
